## CatBoost for Time Series Forecasting
#### CatBoost with Advanced Feature Engineering for Multivariate Time-Series
- **AVAILABLE** for multivariate time-series with exceptional categorical feature handling
- Superior performance on categorical data without preprocessing
- Built-in overfitting detection and GPU acceleration support


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import catboost as cb
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [2]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']


# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []


In [3]:
# Feature Engineering Functions optimized for CatBoost

def create_time_features(df):
    """Create comprehensive time-based features for CatBoost"""
    df = df.copy()
    
    # Basic time features
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    
    # Cyclical encoding for periodicity
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['hour_of_week'] = df['dayofweek'] * 24  # Simulated hour feature
    
    # Additional categorical time features (perfect for CatBoost)
    df['season'] = df['month'].apply(lambda x: 
        'spring' if x in [3,4,5] else
        'summer' if x in [6,7,8] else
        'fall' if x in [9,10,11] else 'winter')
    
    df['month_name'] = df['date'].dt.strftime('%B')
    df['day_name'] = df['date'].dt.strftime('%A')
    df['week_type'] = df['dayofweek'].apply(lambda x: 'weekday' if x < 5 else 'weekend')
    
    # Time-based business features
    df['days_from_start'] = (df['date'] - df['date'].min()).dt.days
    df['days_to_month_end'] = (df['date'] + pd.offsets.MonthEnd(0) - df['date']).dt.days
    
    return df

def create_lag_features(df, target_col='sales', lags=[1, 2, 3, 7, 14, 21, 28]):
    """Create comprehensive lagged features"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for lag in lags:
        # Basic lag
        df[f'{target_col}_lag_{lag}'] = df.groupby('store_menu_id')[target_col].shift(lag)
        
        # Lag differences
        if lag > 1:
            df[f'{target_col}_lag_diff_{lag}'] = (
                df.groupby('store_menu_id')[target_col].shift(lag) - 
                df.groupby('store_menu_id')[target_col].shift(lag*2)
            )
            
        # Lag ratios
        if lag > 1:
            df[f'{target_col}_lag_ratio_{lag}'] = (
                df.groupby('store_menu_id')[target_col].shift(lag) / 
                (df.groupby('store_menu_id')[target_col].shift(lag*2) + 0.001)
            )
    
    return df

def create_rolling_features(df, target_col='sales', windows=[3, 7, 14, 28]):
    """Create comprehensive rolling window features"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    for window in windows:
        # Rolling statistics
        group_rolling = df.groupby('store_menu_id')[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1))
        
        df[f'{target_col}_rolling_mean_{window}'] = group_rolling.mean()
        df[f'{target_col}_rolling_std_{window}'] = group_rolling.std().fillna(0)
        df[f'{target_col}_rolling_min_{window}'] = group_rolling.min()
        df[f'{target_col}_rolling_max_{window}'] = group_rolling.max()
        df[f'{target_col}_rolling_median_{window}'] = group_rolling.median()
        df[f'{target_col}_rolling_sum_{window}'] = group_rolling.sum()
        
        # Advanced rolling features
        df[f'{target_col}_rolling_skew_{window}'] = group_rolling.skew().fillna(0)
        df[f'{target_col}_rolling_kurt_{window}'] = group_rolling.kurt().fillna(0)
        
        # Rolling trends
        df[f'{target_col}_rolling_trend_{window}'] = (
            df[f'{target_col}_rolling_mean_{window}'] - 
            df.groupby('store_menu_id')[f'{target_col}_rolling_mean_{window}'].shift(window)
        ).fillna(0)
        
        # Coefficient of variation
        df[f'{target_col}_rolling_cv_{window}'] = (
            df[f'{target_col}_rolling_std_{window}'] / 
            (df[f'{target_col}_rolling_mean_{window}'] + 0.001)
        )
        
        # Rolling percentiles
        df[f'{target_col}_rolling_q25_{window}'] = group_rolling.quantile(0.25)
        df[f'{target_col}_rolling_q75_{window}'] = group_rolling.quantile(0.75)
        
        # Rolling momentum
        df[f'{target_col}_rolling_momentum_{window}'] = (
            df[target_col] - df[f'{target_col}_rolling_mean_{window}']
        )
    
    return df

def create_expanding_features(df, target_col='sales'):
    """Create expanding window features"""
    df = df.copy()
    df = df.sort_values(['store_menu_id', 'date']).reset_index(drop=True)
    
    # Expanding statistics
    expanding_stats = df.groupby('store_menu_id')[target_col].expanding()
    df[f'{target_col}_expanding_mean'] = expanding_stats.mean().reset_index(0, drop=True)
    df[f'{target_col}_expanding_std'] = expanding_stats.std().fillna(0).reset_index(0, drop=True)
    df[f'{target_col}_expanding_min'] = expanding_stats.min().reset_index(0, drop=True)
    df[f'{target_col}_expanding_max'] = expanding_stats.max().reset_index(0, drop=True)
    df[f'{target_col}_expanding_median'] = expanding_stats.median().reset_index(0, drop=True)
    df[f'{target_col}_expanding_sum'] = expanding_stats.sum().reset_index(0, drop=True)
    df[f'{target_col}_expanding_count'] = expanding_stats.count().reset_index(0, drop=True)
    
    # Expanding ratios
    df[f'{target_col}_vs_expanding_mean'] = (
        df[target_col] / (df[f'{target_col}_expanding_mean'] + 0.001)
    )
    
    return df

def create_interaction_features(df):
    """Create interaction features optimized for CatBoost"""
    df = df.copy()
    
    # Store-time interactions
    df['store_dayofweek'] = df['store'] + "_" + df['dayofweek'].astype(str)
    df['store_month'] = df['store'] + "_" + df['month'].astype(str)
    df['store_season'] = df['store'] + "_" + df['season']
    df['store_weekend'] = df['store'] + "_" + df['is_weekend'].astype(str)
    
    # Menu-time interactions
    df['menu_dayofweek'] = df['menu'] + "_" + df['dayofweek'].astype(str)
    df['menu_month'] = df['menu'] + "_" + df['month'].astype(str)
    df['menu_season'] = df['menu'] + "_" + df['season']
    
    # Complex interactions
    df['store_menu_dayofweek'] = df['store'] + "_" + df['menu'] + "_" + df['dayofweek'].astype(str)
    df['store_menu_month'] = df['store'] + "_" + df['menu'] + "_" + df['month'].astype(str)
    
    return df

def create_target_encoding(df, categorical_cols, target_col='sales'):
    """Create target encoding features (CatBoost handles this internally, but manual encoding can help)"""
    df = df.copy()
    
    for col in categorical_cols:
        if col in df.columns:
            # Basic target encoding with smoothing
            target_mean_global = df[target_col].mean()
            target_stats = df.groupby(col)[target_col].agg(['mean', 'count', 'std']).fillna(0)
            
            # Smoothed target encoding
            smooth_factor = 10
            target_stats['smoothed_mean'] = (
                (target_stats['mean'] * target_stats['count'] + target_mean_global * smooth_factor) /
                (target_stats['count'] + smooth_factor)
            )
            
            df[f'{col}_target_mean'] = df[col].map(target_stats['mean'])
            df[f'{col}_target_std'] = df[col].map(target_stats['std'])
            df[f'{col}_target_count'] = df[col].map(target_stats['count'])
            df[f'{col}_target_smoothed'] = df[col].map(target_stats['smoothed_mean'])
            
            # Time-based target encoding
            for time_col in ['dayofweek', 'month', 'season']:
                if time_col in df.columns:
                    time_target_mean = df.groupby([col, time_col])[target_col].mean()
                    df[f'{col}_{time_col}_target_mean'] = df.set_index([col, time_col]).index.map(
                        time_target_mean).fillna(df[f'{col}_target_mean'])
    
    return df


In [4]:
def prepare_features(df):
    """Prepare all features optimized for CatBoost"""
    df = df.copy()
    
    # Time features
    df = create_time_features(df)
    
    # Interaction features
    df = create_interaction_features(df)
    
    # Lag features
    df = create_lag_features(df)
    
    # Rolling features
    df = create_rolling_features(df)
    
    # Expanding features
    df = create_expanding_features(df)
    
    # Target encoding for categorical features
    categorical_cols = ['store', 'menu']
    df = create_target_encoding(df, categorical_cols)
    
    return df

def get_feature_columns(df):
    """Get feature columns for CatBoost model"""
    exclude_cols = [
        'date', 'store_menu_id', 'sales', 'date_ordinal'
    ]
    
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    # Identify categorical columns (CatBoost excels with these)
    categorical_cols = [
        'store', 'menu', 'season', 'month_name', 'day_name', 'week_type',
        'store_dayofweek', 'store_month', 'store_season', 'store_weekend',
        'menu_dayofweek', 'menu_month', 'menu_season',
        'store_menu_dayofweek', 'store_menu_month'
    ]
    categorical_features = [col for col in categorical_cols if col in feature_cols]
    
    return feature_cols, categorical_features


In [5]:
def predict_with_catboost(train_df, test_df, sid):
    """Predict using CatBoost for a specific store_menu_id"""
    # Filter data for specific store_menu_id
    train_sid = train_df[train_df['store_menu_id'] == sid].copy()
    test_sid = test_df[test_df['store_menu_id'] == sid].copy()
    
    if len(train_sid) == 0 or len(test_sid) == 0:
        raise ValueError(f"No data found for {sid}")
    
    # Combine and sort data
    combined_data = pd.concat([train_sid, test_sid], ignore_index=True)
    combined_data = combined_data.sort_values('date').reset_index(drop=True)
    
    # Prepare features
    combined_data = prepare_features(combined_data)
    
    # Get feature columns
    feature_cols, categorical_features = get_feature_columns(combined_data)
    
    # Split back to train and test
    train_end_idx = len(train_sid)
    train_features = combined_data.iloc[:train_end_idx]
    test_features = combined_data.iloc[train_end_idx:]
    
    # Get last 28 days for prediction input
    input_end_ordinal = test_features['date_ordinal'].max()
    prediction_input = combined_data[
        (combined_data['date_ordinal'] <= input_end_ordinal) & 
        (combined_data['date_ordinal'] > input_end_ordinal - 28)
    ].copy()
    
    if len(prediction_input) != 28:
        raise ValueError(f"{sid} does not have exactly 28 days of input data.")
    
    # Prepare training data (use data before the test period)
    train_data_for_model = combined_data[
        combined_data['date_ordinal'] <= input_end_ordinal
    ].copy()
    
    # Remove rows with None values in target
    train_data_for_model = train_data_for_model.dropna(subset=['sales'])
    
    if len(train_data_for_model) < 30:  # Minimum training samples
        # Fallback to simple mean prediction
        recent_sales = prediction_input['sales'].dropna()
        recent_mean = recent_sales.tail(7).mean() if len(recent_sales) > 0 else 0
        forecast = np.full(7, recent_mean if not pd.isna(recent_mean) else 0)
    else:
        # Prepare features and target
        X_train = train_data_for_model[feature_cols]
        y_train = train_data_for_model['sales']
        
        # Handle categorical features indices for CatBoost
        cat_feature_indices = [feature_cols.index(col) for col in categorical_features if col in feature_cols]
        
        # CatBoost parameters optimized for time series
        params = {
            'iterations': 1000,
            'learning_rate': 0.05,
            'depth': 6,
            'l2_leaf_reg': 3,
            'model_size_reg': 0.5,
            'rsm': 0.95,
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'od_type': 'Iter',
            'od_wait': 50,
            'verbose': False,
            'allow_writing_files': False,
            'thread_count': -1
        }
        
        # Create CatBoost model
        model = CatBoostRegressor(**params)
        
        # Train model with categorical features
        model.fit(
            X_train, 
            y_train,
            cat_features=cat_feature_indices,
            verbose=False
        )
        
        # Predict next 7 days iteratively
        forecast = []
        current_data = combined_data.copy()
        
        for day in range(7):
            # Create next day data
            next_date = prediction_input['date'].max() + timedelta(days=day+1)
            next_ordinal = input_end_ordinal + day + 1
            
            # Create a new row for prediction
            next_row = prediction_input.iloc[-1:].copy()
            next_row['date'] = next_date
            next_row['date_ordinal'] = next_ordinal
            next_row['sales'] = None  # Unknown target
            
            # Add to current data and recreate features
            temp_data = pd.concat([current_data, next_row], ignore_index=True)
            temp_data = temp_data.sort_values('date').reset_index(drop=True)
            temp_data = prepare_features(temp_data)
            
            # Get the prediction row
            pred_row = temp_data.iloc[-1:]
            
            # Handle missing values in features
            pred_features = pred_row[feature_cols].fillna(0)
            
            # Make prediction
            pred_value = model.predict(pred_features)[0]
            pred_value = max(0, pred_value)  # Ensure non-negative
            
            forecast.append(pred_value)
            
            # Update the prediction row with the predicted value
            temp_data.loc[temp_data.index[-1], 'sales'] = pred_value
            current_data = temp_data.copy()
        
        forecast = np.array(forecast)
    
    # Create forecast dates
    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
    
    return pd.DataFrame({
        'date': forecast_dates,
        'store_menu_id': sid,
        'sales': forecast
    })


In [6]:
def run_recursive_forecasting_catboost(train_df, test_data_list):
    """Run recursive forecasting using CatBoost"""
    all_predictions = []
    
    for i, test_df in enumerate(test_data_list):
        test_df = test_df.copy()
        test_df['date'] = pd.to_datetime(test_df['date'])
        test_df = test_df.sort_values(['store_menu_id', 'date'])
        
        pred_list = []
        store_menu_ids = test_df['store_menu_id'].unique()
        
        for sid in tqdm(store_menu_ids, desc=f"Predicting TEST_{i} with CatBoost"):
            try:
                pred_df = predict_with_catboost(train_df, test_df, sid)
                pred_list.append(pred_df)
                
                # Update train_df: add current test + prediction
                test_part = test_df[test_df['store_menu_id'] == sid]
                train_df = pd.concat([train_df, test_part, pred_df])
                
            except Exception as e:
                print(f"Failed for {sid}: {e}")
                # Create fallback prediction (zero or recent mean)
                test_part = test_df[test_df['store_menu_id'] == sid]
                if len(test_part) > 0:
                    input_end_ordinal = test_part['date_ordinal'].max()
                    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
                    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])
                    
                    # Use recent mean as fallback
                    recent_mean = test_part['sales'].tail(7).mean()
                    fallback_value = recent_mean if not pd.isna(recent_mean) else 0
                    
                    fallback_pred = pd.DataFrame({
                        'date': forecast_dates,
                        'store_menu_id': sid,
                        'sales': np.full(7, fallback_value)
                    })
                    pred_list.append(fallback_pred)
                    
                    # Update train_df with test data and fallback prediction
                    train_df = pd.concat([train_df, test_part, fallback_pred])
        
        if pred_list:
            all_predictions.append(pd.concat(pred_list))
    
    return pd.concat(all_predictions) if all_predictions else pd.DataFrame()


In [7]:
# Prepare test data list
test_data_list = []
for i in range(10):
    test_data_list.append(globals()[f'test_data_{i}'])

# Run CatBoost forecasting
print("Starting CatBoost recursive forecasting...")
final_predictions = run_recursive_forecasting_catboost(train_data.copy(), test_data_list)

print(f"Total predictions generated: {len(final_predictions)}")
if len(final_predictions) > 0:
    print("First few predictions:")
    print(final_predictions.head(10))


Starting CatBoost recursive forecasting...


Predicting TEST_0 with CatBoost:   0%|          | 0/193 [00:00<?, ?it/s]

Failed for 느티나무 셀프BBQ_1인 수저세트: cannot convert the series to <class 'float'>


Predicting TEST_0 with CatBoost:   1%|          | 2/193 [00:00<00:31,  6.05it/s]

Failed for 느티나무 셀프BBQ_BBQ55(단체): cannot convert the series to <class 'float'>


...

Failed for 화담숲카페_아메리카노 ICE: cannot convert the series to <class 'float'>


Predicting TEST_9 with CatBoost:  99%|█████████▉| 192/193 [00:39<00:00,  4.74it/s]

Failed for 화담숲카페_카페라떼 ICE: cannot convert the series to <class 'float'>


Predicting TEST_9 with CatBoost: 100%|██████████| 193/193 [00:39<00:00,  4.83it/s]

Failed for 화담숲카페_현미뻥스크림: cannot convert the series to <class 'float'>
Total predictions generated: 13510
First few predictions:
        date         store_menu_id     sales
0 2024-07-14    느티나무 셀프BBQ_1인 수저세트  7.857143
1 2024-07-15    느티나무 셀프BBQ_1인 수저세트  7.857143
2 2024-07-16    느티나무 셀프BBQ_1인 수저세트  7.857143
3 2024-07-17    느티나무 셀프BBQ_1인 수저세트  7.857143
4 2024-07-18    느티나무 셀프BBQ_1인 수저세트  7.857143
5 2024-07-19    느티나무 셀프BBQ_1인 수저세트  7.857143
6 2024-07-20    느티나무 셀프BBQ_1인 수저세트  7.857143
0 2024-07-14  느티나무 셀프BBQ_BBQ55(단체)  2.000000
1 2024-07-15  느티나무 셀프BBQ_BBQ55(단체)  2.000000
2 2024-07-16  느티나무 셀프BBQ_BBQ55(단체)  2.000000


In [8]:
# Create submission file
def create_submission_file(predictions_df, submission_template, output_path):
    """Create submission file in the required format"""
    
    if len(predictions_df) == 0:
        print("No predictions to create submission file.")
        return None
    
    # Initialize submission with template
    submission_df = submission_template.copy()
    
    # Group predictions by store_menu_id and date
    predictions_pivot = predictions_df.pivot_table(
        index='date', 
        columns='store_menu_id', 
        values='sales', 
        fill_value=0
    )
    
    print(f"Predictions pivot shape: {predictions_pivot.shape}")
    print(f"Submission template shape: {submission_df.shape}")
    
    # Fill submission template with predictions
    for col in submission_df.columns:
        if col in predictions_pivot.columns:
            # Get predictions for this store_menu combination
            pred_values = predictions_pivot[col].values
            if len(pred_values) == len(submission_df):
                submission_df[col] = pred_values
            else:
                print(f"Warning: Mismatch in prediction length for {col}: {len(pred_values)} vs {len(submission_df)}")
                # Fill with available predictions or zeros
                min_len = min(len(pred_values), len(submission_df))
                submission_df.loc[:min_len-1, col] = pred_values[:min_len]
                if min_len < len(submission_df):
                    submission_df.loc[min_len:, col] = 0
        else:
            print(f"Warning: No predictions found for {col}")
            submission_df[col] = 0  # Fill with zeros if no prediction
    
    # Save submission file
    submission_df.to_csv(output_path, index=False)
    print(f"Submission file saved to: {output_path}")
    
    return submission_df

# Create and save submission
if len(final_predictions) > 0:
    submission_result = create_submission_file(
        final_predictions, 
        submission, 
        "./result/catboost_submission.csv"
    )
    
    if submission_result is not None:
        print("\nSubmission file shape:", submission_result.shape)
        print("Sample submission values:")
        print(submission_result.iloc[:5, :5])
        
        # Check for None values
        none_count = submission_result.isna().sum().sum()
        print(f"\nTotal None values in submission: {none_count}")
        
        # Basic statistics
        numeric_cols = submission_result.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print("\nSubmission statistics:")
            print(f"Mean: {submission_result[numeric_cols].mean().mean():.4f}")
            print(f"Std: {submission_result[numeric_cols].std().mean():.4f}")
            print(f"Min: {submission_result[numeric_cols].min().min():.4f}")
            print(f"Max: {submission_result[numeric_cols].max().max():.4f}")
else:
    print("No predictions generated. Please check the model.")


Predictions pivot shape: (70, 193)
Submission template shape: (70, 194)
Submission file saved to: ./result/catboost_submission.csv

Submission file shape: (70, 194)
Sample submission values:
   date  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  느티나무 셀프BBQ_대여료 30,000원  \
0     0            7.857143                   2.0                4.285714   
1     0            7.857143                   2.0                4.285714   
2     0            7.857143                   2.0                4.285714   
3     0            7.857143                   2.0                4.285714   
4     0            7.857143                   2.0                4.285714   

   느티나무 셀프BBQ_대여료 60,000원  
0                2.571429  
1                2.571429  
2                2.571429  
3                2.571429  
4                2.571429  

Total None values in submission: 0

Submission statistics:
Mean: 8.6377
Std: 10.8763
Min: 0.0000
Max: 575.0000


In [9]:
# Model evaluation and performance analysis
def evaluate_model_performance(predictions_df):
    """Comprehensive evaluation of model performance"""
    
    if len(predictions_df) == 0:
        print("No predictions to evaluate.")
        return
        
    print("=== CatBoost Model Performance Summary ===")
    print(f"Total predictions: {len(predictions_df)}")
    print(f"Unique store-menu combinations: {predictions_df['store_menu_id'].nunique()}")
    print(f"Prediction date range: {predictions_df['date'].min()} to {predictions_df['date'].max()}")
    
    print("\n=== Sales Prediction Statistics ===")
    print(predictions_df['sales'].describe())
    
    # Check for negative predictions
    negative_count = (predictions_df['sales'] < 0).sum()
    print(f"\nNegative predictions: {negative_count}")
    
    # Check for None values
    none_count = predictions_df['sales'].isna().sum()
    print(f"None predictions: {none_count}")
    
    print("\n=== Top 10 Store-Menu by Predicted Sales ===")
    top_predictions = predictions_df.groupby('store_menu_id')['sales'].sum().sort_values(ascending=False).head(10)
    for idx, (store_menu, total_sales) in enumerate(top_predictions.items(), 1):
        print(f"{idx:2d}. {store_menu}: {total_sales:.2f}")
    
    print("\n=== Daily Prediction Patterns ===")
    daily_stats = predictions_df.groupby('date')['sales'].agg(['count', 'mean', 'std']).round(2)
    print(daily_stats)
    
    print("\n=== Store-wise Prediction Summary ===")
    if 'store' in predictions_df['store_menu_id'].str.split('_').str[0].values:
        predictions_df_temp = predictions_df.copy()
        predictions_df_temp['store'] = predictions_df_temp['store_menu_id'].str.split('_').str[0]
        store_stats = predictions_df_temp.groupby('store')['sales'].agg(['count', 'mean', 'sum']).round(2)
        print(store_stats)

if len(final_predictions) > 0:
    evaluate_model_performance(final_predictions)


=== CatBoost Model Performance Summary ===
Total predictions: 13510
Unique store-menu combinations: 193
Prediction date range: 2024-07-14 00:00:00 to 2025-05-31 00:00:00

=== Sales Prediction Statistics ===
count    13510.000000
mean         8.682457
std         28.557979
min          0.000000
25%          0.285714
50%          1.285714
75%          4.571429
max        575.000000
Name: sales, dtype: float64

Negative predictions: 0
None predictions: 0

=== Top 10 Store-Menu by Predicted Sales ===
 1. 화담숲주막_해물파전: 8078.00
 2. 포레스트릿_꼬치어묵: 5560.00
 3. 카페테리아_단체식 18000(신): 3954.00
 4. 미라시아_브런치(대인) 주말: 3464.00
 5. 포레스트릿_생수: 3373.00
 6. 포레스트릿_떡볶이: 3268.00
 7. 화담숲카페_아메리카노 ICE: 3174.00
 8. 카페테리아_수제 등심 돈까스: 2886.00
 9. 카페테리아_단체식 13000(신): 2756.00
10. 미라시아_브런치(대인) 주중: 2693.00

=== Daily Prediction Patterns ===
            count  mean    std
date                          
2024-07-14    193  4.17   8.16
2024-07-15    193  4.17   8.16
2024-07-16    193  4.17   8.16
2024-07-17    193  4.17   8.16
2024

In [10]:
# Feature importance analysis with CatBoost
def analyze_feature_importance_catboost(sample_data):
    """Analyze feature importance using CatBoost"""
    
    print("=== CatBoost Feature Importance Analysis ===")
    print("Training a sample model to analyze feature importance...")
    
    try:
        # Prepare sample data
        sample_prepared = prepare_features(sample_data)
        feature_cols, categorical_features = get_feature_columns(sample_prepared)
        
        # Remove rows with None target values
        sample_clean = sample_prepared.dropna(subset=['sales'])
        
        if len(sample_clean) < 100:
            print("Not enough clean data for feature importance analysis.")
            return
        
        # Prepare features and target
        X = sample_clean[feature_cols].fillna(0)
        y = sample_clean['sales']
        
        # Simple train-test split
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
        
        # Handle categorical features indices
        cat_feature_indices = [feature_cols.index(col) for col in categorical_features if col in feature_cols]
        
        # Train CatBoost model for feature importance
        params = {
            'iterations': 500,
            'learning_rate': 0.1,
            'depth': 6,
            'verbose': False,
            'allow_writing_files': False,
            'random_seed': 42
        }
        
        model = CatBoostRegressor(**params)
        model.fit(
            X_train, 
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_val, y_val),
            verbose=False
        )
        
        # Get feature importance
        feature_importance = model.get_feature_importance()
        feature_importance_df = pd.DataFrame({
            'feature': feature_cols,
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print("\nTop 20 Most Important Features:")
        print(feature_importance_df.head(20).to_string(index=False))
        
        # Get feature importance by type
        print("\nTop 10 Categorical Features:")
        cat_importance = feature_importance_df[
            feature_importance_df['feature'].isin(categorical_features)
        ].head(10)
        print(cat_importance.to_string(index=False))
        
        # Validation performance
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        
        print(f"\nSample Model Performance:")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        
        # CatBoost specific metrics
        print(f"\nCatBoost Model Info:")
        print(f"Number of trees: {model.tree_count_}")
        print(f"Number of categorical features: {len(cat_feature_indices)}")
        print(f"Best iteration: {model.get_best_iteration()}")
        
    except Exception as e:
        print(f"Feature importance analysis failed: {e}")

# Run feature importance analysis on a sample
if len(train_data) > 1000:
    sample_size = min(5000, len(train_data))
    sample_data = train_data.sample(n=sample_size, random_state=42)
    analyze_feature_importance_catboost(sample_data)


=== CatBoost Feature Importance Analysis ===
Training a sample model to analyze feature importance...
Feature importance analysis failed: cannot convert the series to <class 'float'>


In [12]:
# Hyperparameter tuning for CatBoost (optional)
def tune_catboost_hyperparameters(sample_data, n_trials=50):
    """Hyperparameter tuning using Optuna (if available)"""
    
    try:
        import optuna
        from optuna.samplers import TPESampler
        
        print("=== CatBoost Hyperparameter Tuning ===")
        
        # Prepare data
        sample_prepared = prepare_features(sample_data)
        feature_cols, categorical_features = get_feature_columns(sample_prepared)
        sample_clean = sample_prepared.dropna(subset=['sales'])
        
        if len(sample_clean) < 1000:
            print("Not enough data for hyperparameter tuning.")
            return None
        
        X = sample_clean[feature_cols].fillna(0)
        y = sample_clean['sales']
        
        # Train-validation split
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
        
        cat_feature_indices = [feature_cols.index(col) for col in categorical_features if col in feature_cols]
        
        def objective(trial):
            params = {
                'iterations': trial.suggest_int('iterations', 500, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'depth': trial.suggest_int('depth', 4, 10),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'model_size_reg': trial.suggest_float('model_size_reg', 0.1, 1.0),
                'rsm': trial.suggest_float('rsm', 0.5, 1.0),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'verbose': False,
                'allow_writing_files': False,
                'random_seed': 42
            }
            
            model = CatBoostRegressor(**params)
            model.fit(
                X_train,
                y_train,
                cat_features=cat_feature_indices,
                eval_set=(X_val, y_val),
                verbose=False,
                early_stopping_rounds=50
            )
            
            y_pred = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            return rmse
        
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=42)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        print(f"\nBest parameters found:")
        for key, value in study.best_params.items():
            print(f"{key}: {value}")
        print(f"\nBest RMSE: {study.best_value:.4f}")
        
        return study.best_params
        
    except ImportError:
        print("Optuna not available. Install with: pip install optuna")
        return None
    except Exception as e:
        print(f"Hyperparameter tuning failed: {e}")
        return None

# Uncomment to run hyperparameter tuning
# if len(train_data) > 2000:
#     sample_data = train_data.sample(n=2000, random_state=42)
#     best_params = tune_catboost_hyperparameters(sample_data, n_trials=20)
